# SDXL Phase 1 — Text-to-Image Generation on Colab

Self-contained notebook: loads SDXL Base, generates portraits, computes metrics, logs to MLflow.

**Runtime:** GPU (T4, free tier works)

## Cell 1 — Environment setup

In [ ]:
!pip install -q diffusers==0.30.0 transformers accelerate safetensors invisible_watermark mlflow ftfy
!pip install -q git+https://github.com/openai/CLIP.git

import os
os.makedirs('/content/outputs', exist_ok=True)

# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Load SDXL Base

In [ ]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
).to('cuda')

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Memory-efficient attention (xformers if available, else PyTorch SDPA)
try:
    pipe.enable_xformers_memory_efficient_attention()
    print('xformers attention enabled')
except Exception as e:
    print(f'xformers not available ({e}), using SDPA fallback')

print('SDXL Base loaded')

## Cell 3 — Load CLIP for metrics

In [ ]:
import clip

clip_model, clip_preprocess = clip.load('ViT-B/32', device='cuda')
clip_model.eval()
print('CLIP ViT-B/32 loaded')

## Cell 4 — Metrics (CLIP similarity, sharpness, diversity)

In [ ]:
import numpy as np
from PIL import Image
import torch.nn.functional as F
from scipy.ndimage import convolve


def clip_similarity(pil_image, prompt):
    """Cosine similarity between image and text prompt in CLIP space."""
    image_input = clip_preprocess(pil_image).unsqueeze(0).to('cuda')
    text_tokens = clip.tokenize([prompt], truncate=True).to('cuda')
    with torch.no_grad():
        image_features = clip_model.encode_image(image_input)
        text_features = clip_model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    return (image_features * text_features).sum(dim=-1).item()


def sharpness(pil_image):
    """Laplacian variance on greyscale — higher = sharper."""
    arr = np.array(pil_image.convert('L'), dtype=np.float64)
    kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
    return float(convolve(arr, kernel).var())


def pixel_diversity(pil_image):
    """Std dev across all pixels — near-zero = mode collapse."""
    arr = np.array(pil_image, dtype=np.float32) / 255.0
    return float(arr.std())


print('Metrics ready: clip_similarity, sharpness, pixel_diversity')

## Cell 5 — MLflow setup

In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:////content/mlflow.db')
mlflow.set_experiment('sdxl_phase1')
print('MLflow tracking: /content/mlflow.db → experiment sdxl_phase1')

## Cell 6 — Generation function

In [ ]:
import time


def generate(prompt, negative_prompt='', steps=30, guidance=7.5, seed=42,
             width=1024, height=1024):
    """Generate an image with SDXL. Returns (PIL.Image, elapsed_seconds, vram_peak_gb)."""
    torch.cuda.reset_peak_memory_stats()
    gen = torch.Generator('cuda').manual_seed(seed)

    start = time.perf_counter()
    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
        width=width,
        height=height,
    ).images[0]
    elapsed = time.perf_counter() - start
    vram_peak = torch.cuda.max_memory_allocated() / 1e9

    return image, elapsed, vram_peak


print('generate() ready')

## Cell 7 — Run portrait prompts + log to MLflow

In [ ]:
PROMPTS = [
    'portrait of a young woman with red hair, photorealistic',
    'portrait of an elderly man with white beard and long hair, photorealistic',
    'portrait of a middle-aged man with blonde hair and blue eyes, photorealistic',
]

NEGATIVE = 'blurry, low quality, cartoon, painting, sketch, deformed, disfigured, watermark, text'

results = []

for prompt in PROMPTS:
    print(f'\nGenerating: "{prompt}"')

    with mlflow.start_run(run_name=prompt[:40]):
        image, elapsed, vram_peak = generate(
            prompt, negative_prompt=NEGATIVE, steps=30, guidance=7.5, seed=42,
        )

        # Compute metrics
        clip_sim = clip_similarity(image, prompt)
        sharp = sharpness(image)
        diversity = pixel_diversity(image)

        # Save artifact
        filename = prompt.replace(' ', '_').replace(',', '')[:60] + '.png'
        path = f'/content/outputs/{filename}'
        image.save(path)

        # Log to MLflow
        mlflow.log_params({
            'prompt': prompt,
            'negative_prompt': NEGATIVE,
            'steps': 30,
            'guidance': 7.5,
            'seed': 42,
            'model': 'SDXL-Base-1.0',
            'scheduler': 'DPMSolverMultistep',
        })
        mlflow.log_metrics({
            'clip_similarity': clip_sim,
            'sharpness': sharp,
            'pixel_diversity': diversity,
            'generation_time_s': elapsed,
            'vram_peak_gb': vram_peak,
        })
        mlflow.log_artifact(path)

        print(f'  CLIP sim: {clip_sim:.4f}  sharp: {sharp:.1f}  div: {diversity:.3f}  time: {elapsed:.1f}s  vram: {vram_peak:.1f}GB')
        results.append((prompt, image, clip_sim, sharp, diversity, elapsed))

print(f'\nGenerated {len(results)} images → /content/outputs/')

## Cell 8 — Parameter sweep (optional)

Same prompt, different `guidance_scale` and `steps` → find the best combo for Phase 2.

In [ ]:
SWEEP_PROMPT = 'portrait of a young woman with red hair, photorealistic'

sweep_results = []

for guidance in [5.0, 7.5, 10.0, 12.0]:
    for steps in [20, 30, 50]:
        print(f'guidance={guidance}  steps={steps}')

        with mlflow.start_run(run_name=f'sweep_g{guidance}_s{steps}'):
            image, elapsed, vram_peak = generate(
                SWEEP_PROMPT, negative_prompt=NEGATIVE,
                steps=steps, guidance=guidance, seed=42,
            )
            clip_sim = clip_similarity(image, SWEEP_PROMPT)
            sharp = sharpness(image)
            diversity = pixel_diversity(image)

            filename = f'sweep_g{guidance}_s{steps}.png'
            path = f'/content/outputs/{filename}'
            image.save(path)

            mlflow.log_params({
                'prompt': SWEEP_PROMPT,
                'steps': steps,
                'guidance': guidance,
                'seed': 42,
                'sweep': True,
            })
            mlflow.log_metrics({
                'clip_similarity': clip_sim,
                'sharpness': sharp,
                'pixel_diversity': diversity,
                'generation_time_s': elapsed,
            })
            mlflow.log_artifact(path)

            sweep_results.append((guidance, steps, image, clip_sim, sharp))

print(f'\nSweep complete: {len(sweep_results)} combos')

## Cell 9 — Display all generated images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6))
if len(results) == 1:
    axes = [axes]
for ax, (prompt, image, clip_sim, sharp, diversity, elapsed) in zip(axes, results):
    ax.imshow(image)
    ax.axis('off')
    short = prompt[:40] + ('...' if len(prompt) > 40 else '')
    ax.set_title(f'{short}\nCLIP: {clip_sim:.3f}  sharp: {sharp:.0f}  time: {elapsed:.1f}s', fontsize=10)
plt.tight_layout()
plt.show()

## Cell 10 — Launch MLflow UI (optional)

Use pyngrok to expose the MLflow UI publicly.

In [ ]:
# Install ngrok client and start MLflow UI in background
!pip install -q pyngrok

# Kill any existing mlflow process
!pkill -f 'mlflow ui' || true

# Start MLflow UI
import subprocess
mlflow_proc = subprocess.Popen(
    ['mlflow', 'ui', '--backend-store-uri', 'sqlite:////content/mlflow.db', '--port', '5000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

import time
time.sleep(3)

# Expose via ngrok (free tier — needs authtoken from ngrok.com for longer sessions)
from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_TOKEN_HERE')  # optional, for persistent tunnels
public_url = ngrok.connect(5000)
print(f'MLflow UI: {public_url}')